# 01 — Raw ADLS → Bronze

## Student Demo

Goal:

```text
ADLS Raw Files
     ↓
Read Raw Data
     ↓
Bronze Delta Table
```

This is intentionally simple. No Auto Loader yet.

Replace only `RAW_PATH` with the ADLS folder containing the Lab 1 files.


In [ ]:
from pyspark.sql import functions as F

RAW_PATH = "abfss://retail@retaildatalake1222.dfs.core.windows.net/raw/"

CATALOG = "retail_catalog2"
BRONZE_SCHEMA = "bronze"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")

print("Raw path:", RAW_PATH)


## 1. Read the raw files

If Lab 1 contains different formats, read each format separately.

Example:

```text
raw/
├── customers.csv
├── orders.json
└── products.parquet
```


In [ ]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH + "customers.csv")
)

orders = spark.read.json(RAW_PATH + "orders.json")

products = spark.read.parquet(RAW_PATH + "products.parquet")

display(customers)
display(orders)
display(products)


## 2. Add Bronze metadata

Bronze should preserve the source data with only minimal technical metadata.


In [ ]:
customers_bronze = (
    customers
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("customers.csv"))
)

orders_bronze = (
    orders
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("orders.json"))
)

products_bronze = (
    products
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit("products.parquet"))
)


## 3. Write Bronze Delta tables


In [ ]:
(
    customers_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
)

(
    orders_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.orders")
)

(
    products_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOG}.{BRONZE_SCHEMA}.products")
)

print("Bronze tables created")


In [ ]:
%sql
SHOW TABLES IN retail_catalog.bronze;


## Student explanation

> Bronze is the first Delta representation of the source data. We keep the source structure and add technical metadata. Business transformations belong in Silver.
